In [1]:
from pyspark.sql import SparkSession
from delta import *

In [2]:
from pyspark.sql import SparkSession

builder = SparkSession.builder \
    .appName("Local_Lakehouse_70GB_GPU") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.driver.memory", "4g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "1g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "2000") \
    .config("spark.sql.caseSensitive", "true")

In [3]:
spark = configure_spark_with_delta_pip(builder).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/11 15:49:22 WARN Utils: Your hostname, LAPTOP-MSL0MUJB, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/11 15:49:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/harsh/wsl_venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/harsh/.ivy2.5.2/cache
The jars for the packages stored in: /home/harsh/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-773bc737-0506-43ae-a806-cb1e15d03c0c;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache

In [4]:
sentimentTable = spark.read \
    .format("parquet") \
    .load("/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/vaderSentimentTable/")

In [5]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [6]:
sentimentTable=sentimentTable.withColumn("splitted_text",array_distinct(split(lower(col("commit.record.text")),r"\s+")))

In [7]:
sentimentTable=sentimentTable.withColumn("exploded_text",explode(col("splitted_text")))

In [11]:
# 1. Convert the microsecond integer into a real Spark Timestamp
df_with_time = sentimentTable.withColumn(
    "real_timestamp", 
    (col("time_us") / 1000000).cast("timestamp")
)

# 2. Group by a 2-hour Tumbling Window AND the specific word
windowed_sentiment = df_with_time.groupBy(
    window(col("real_timestamp"), "2 hours"),
    col("exploded_text")
).agg(
    avg("vader_sentiment_score").alias("avg_sentiment")
)

In [12]:
windowed_sentiment.show(n=100,truncate=False)

+------------------------------------------+---------------------------+----------------------+
|window                                    |exploded_text              |avg_sentiment         |
+------------------------------------------+---------------------------+----------------------+
|{2026-04-03 08:00:00, 2026-04-03 10:00:00}|"age                       |-0.299699991941452    |
|{2026-04-03 08:00:00, 2026-04-03 10:00:00}|"arizona                   |-0.051600001752376556 |
|{2026-04-03 08:00:00, 2026-04-03 10:00:00}|"assault                   |-0.8082000017166138   |
|{2026-04-03 08:00:00, 2026-04-03 10:00:00}|"despiertos"               |0.0                   |
|{2026-04-03 08:00:00, 2026-04-03 10:00:00}|"for                       |0.09966666499773662   |
|{2026-04-03 08:00:00, 2026-04-03 10:00:00}|"look                      |0.4339333275953929    |
|{2026-04-03 08:00:00, 2026-04-03 10:00:00}|"mais                      |-0.12600000202655792  |
|{2026-04-03 08:00:00, 2026-04-03 10:00: